# Lab 18: Spark MLlib

**Module 18** | Read `notes/18-spark-mllib.pdf` before running this notebook.

Build a Spark ML pipeline on the credit-fraud sample and compare fit time with scikit-learn on the same rows.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Distributed ML with PySpark MLlib

Credit-card fraud detection is a classic **imbalanced classification** problem.
The sample dataset contains anonymized features **V1 to V28** (PCA components) plus `Amount`, `Time`, and a binary `Class` label.

Spark MLlib runs pipelines across a cluster (here: `local[1]` on one machine) using DataFrame operations.
We build a **Pipeline** with `VectorAssembler` then `RandomForestClassifier`, then compare fit time and accuracy against **scikit-learn** on the same rows.

If `credit_fraud_sample.parquet` is missing, the lab falls back to the CSV version automatically.


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **Apache Spark** | A distributed computing engine for large datasets. Processes data in parallel across many machines (or one local thread pool). |
| **Spark DataFrame** | A tabular dataset distributed across partitions. Transformations are lazy: Spark builds a plan and runs it when an action (like `count()`) is called. |
| **Spark partition** | A chunk of rows that Spark processes on one worker at a time. More partitions enable parallelism on clusters; too many small partitions add overhead. |
| **MLlib Pipeline** | Chains feature engineering and model training steps. Calling `pipeline.fit(train)` runs every stage in order on the training DataFrame. |
| **VectorAssembler** | Combines multiple numeric columns (V1..V28) into one feature vector column Spark models expect. |
| **RandomForestClassifier** | An ensemble of decision trees. Each tree votes; majority vote is the prediction. |
| **Imbalanced classification** | One class (fraud) is much rarer than the other (legitimate). Accuracy alone can be misleading; inspect confusion counts. |
| **Parquet** | A columnar file format Spark reads efficiently. Faster and smaller than CSV for analytics. |

Refer back to this table whenever a term appears in code or output.


### Load data and inspect schema

**What this cell does:** Starts a local Spark session, loads the credit-fraud sample (Parquet or CSV), and prints schema, sample rows, and class balance.

**Why we do this:** Always inspect data before training. Class imbalance (very few fraud rows) affects how you interpret accuracy.


**What to look for in the output**

You should see:
- A row count (roughly a few thousand rows)
- A printed schema showing V1..V28, Amount, Time, Class
- Three sample rows
- Class balance showing Class 0 (legit) dominates Class 1 (fraud) by a large margin


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime, create_spark, stop_spark

configure_runtime(quiet=True)

from pyspark.sql import functions as F

# Paths to the credit fraud sample (Parquet preferred, CSV fallback)
parquet_path = ROOT / "datasets" / "credit_fraud_sample.parquet"
csv_path = ROOT / "datasets" / "credit_fraud_sample.csv"
feature_cols = [f"V{i}" for i in range(1, 29)]  # V1 through V28

# create_spark("Lab18") starts a JVM + Spark session (local[1] = one worker thread)
spark = create_spark("Lab18")
if parquet_path.exists():
    df = spark.read.parquet(str(parquet_path))
    source_name = parquet_path.name
else:
    df = spark.read.option("header", True).option("inferSchema", True).csv(str(csv_path))
    source_name = csv_path.name

# count() is an action: Spark executes the read plan and returns total rows
row_count = df.count()
print(f"Loaded {source_name}: {row_count:,} rows")
print()
print("Schema:")
df.printSchema()
print("Sample rows (first 3):")
df.select("Time", "Amount", "V1", "V2", "V28", "Class").show(3, truncate=False)

# Group by Class to see fraud vs legit counts (imbalance check)
balance = df.groupBy("Class").count().orderBy("Class").collect()
print("Class balance:")
for row in balance:
    label = "legit" if row["Class"] == 0 else "fraud"
    pct = 100.0 * row["count"] / row_count
    print(f"  Class {int(row['Class'])} ({label}): {row['count']:,} rows ({pct:.2f}%)")


### Train with Spark MLlib

**What this cell does:** Builds a Pipeline (VectorAssembler + RandomForest), splits 80/20, fits on train, evaluates accuracy on test, and prints confusion-style counts.

**Why we do this:** Spark MLlib expresses the full workflow as DataFrame transformations. On a cluster with millions of rows, the same pipeline scales out across **partitions** automatically.


**What to look for in the output**

You should see:
- Train and test row counts (about 80% / 20% of total)
- Spark fit time in seconds
- Spark test accuracy (often high because legit class dominates)
- A confusion-style table of actual Class vs predicted counts


In [ ]:
import time

from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler

# Stage 1: pack V1..V28 into a single "features" vector column
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
# Stage 2: 50-tree random forest classifier
rf = RandomForestClassifier(
    labelCol="Class",
    featuresCol="features",
    numTrees=50,
    maxDepth=8,
    seed=42,
)
pipeline = Pipeline(stages=[assembler, rf])

# randomSplit shuffles and splits; seed=42 for reproducibility
train, test = df.randomSplit([0.8, 0.2], seed=42)
train_count = train.count()
test_count = test.count()
print(f"Train rows: {train_count:,}, test rows: {test_count:,}")

t0 = time.perf_counter()
# fit() runs assembler.transform then trains the forest on the train partition(s)
spark_model = pipeline.fit(train)
spark_fit_sec = time.perf_counter() - t0

predictions = spark_model.transform(test)
evaluator = MulticlassClassificationEvaluator(
    labelCol="Class", predictionCol="prediction", metricName="accuracy"
)
spark_accuracy = evaluator.evaluate(predictions)

print(f"Spark fit time: {spark_fit_sec:.3f} s")
print(f"Spark test accuracy: {spark_accuracy:.4f}")
print()
print("Confusion-style counts (actual Class vs predicted):")
conf_rows = (
    predictions.groupBy("Class", "prediction")
    .count()
    .orderBy("Class", "prediction")
    .collect()
)
print(f"{'Actual':>8} | {'Predicted':>10} | {'Count':>8}")
print("-" * 32)
for row in conf_rows:
    actual = int(row["Class"])
    predicted = int(row["prediction"])
    print(f"{actual:>8} | {predicted:>10} | {row['count']:>8}")

# Stop Spark to free JVM memory before the sklearn comparison
stop_spark(spark)


### Compare with scikit-learn on the same features

**What this cell does:** Loads the same data into pandas, trains an equivalent Random Forest with sklearn, and prints a side-by-side timing and accuracy table.

**Why we do this:** On a small local sample, sklearn often wins on speed because Spark pays JVM startup and distributed planning overhead. Spark's advantage appears at much larger scale.


**What to look for in the output**

You should see:
- Pandas train/test row counts matching Spark's split sizes closely
- sklearn fit time (usually faster than Spark on this small data)
- sklearn test accuracy (close to but not identical to Spark)
- A confusion matrix for sklearn
- SPARK vs SKLEARN TIMING TABLE with fit times, accuracies, and fit-time ratio


In [ ]:
import time

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

if parquet_path.exists():
    pdf = pd.read_parquet(parquet_path)
else:
    pdf = pd.read_csv(csv_path)

X = pdf[feature_cols]
y = pdf["Class"]
# stratify=y keeps the fraud/legit ratio similar in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Pandas train rows: {len(X_train):,}, test rows: {len(X_test):,}")

t0 = time.perf_counter()
sk_model = RandomForestClassifier(
    n_estimators=50, max_depth=8, random_state=42, n_jobs=-1
)
sk_model.fit(X_train, y_train)
sklearn_fit_sec = time.perf_counter() - t0

sk_preds = sk_model.predict(X_test)
sk_accuracy = accuracy_score(y_test, sk_preds)
cm = confusion_matrix(y_test, sk_preds)

print(f"sklearn fit time: {sklearn_fit_sec:.3f} s")
print(f"sklearn test accuracy: {sk_accuracy:.4f}")
print()
print("sklearn confusion matrix (rows=actual, cols=predicted):")
print(f"              pred 0    pred 1")
print(f"  actual 0    {cm[0, 0]:6d}    {cm[0, 1]:6d}")
print(f"  actual 1    {cm[1, 0]:6d}    {cm[1, 1]:6d}")

print()
print("=" * 55)
print("SPARK vs SKLEARN TIMING TABLE")
print("=" * 55)
print(f"{'Metric':<22} | {'Spark':>12} | {'sklearn':>12}")
print("-" * 55)
print(f"{'Fit time (seconds)':<22} | {spark_fit_sec:>12.3f} | {sklearn_fit_sec:>12.3f}")
print(f"{'Test accuracy':<22} | {spark_accuracy:>12.4f} | {sk_accuracy:>12.4f}")
ratio = spark_fit_sec / sklearn_fit_sec if sklearn_fit_sec > 0 else float("inf")
print(f"{'Fit-time ratio Spark/sklearn':<22} | {ratio:>12.2f}x | {'1.00x':>12}")


## Spark vs scikit-learn

On a **small local sample** (~5k rows), scikit-learn usually **fits faster** because Spark pays JVM startup, DataFrame planning, and serialization overhead.
Spark's advantage appears at **millions+ of rows** on a cluster where data does not fit in memory on one machine.

Accuracies should be **close but not identical**: different random splits (Spark `randomSplit` vs sklearn `train_test_split`) and implementation details cause small differences.

**When to use which:**

- **sklearn:** prototyping, datasets that fit in RAM, single-machine speed.
- **Spark MLlib:** large-scale ETL + training in one pipeline, data already in a lake/warehouse, distributed feature engineering.
